# SageMaker: TF‑IDF + XGBoost (Text Classification)

**What this notebook does**
- Reads one or more CSV files from S3 (columns: `text,label`)
- Minimal cleaning with `gensim` (good for TF‑IDF; avoid for BERT)
- Stratified train/val/test split
- TF‑IDF (fit on train only) → XGBoost (weighted samples for imbalance)
- Saves artifacts locally **and** uploads them to S3

**How to use**
1. Set the `CONFIG` cell (S3 bucket/prefix, region, optional multi-file pattern).
2. Run all cells. If your dataset is large, consider enabling `CHUNKED_READ`.

**Assumptions**
- Running inside SageMaker Studio or a Notebook Instance with an execution role.
- The role has `s3:GetObject`/`s3:ListBucket`/`s3:PutObject` for the chosen bucket.

In [ ]:
#!pip install -q boto3 s3fs gensim scikit-learn xgboost pandas numpy joblib
import os, io, sys, json, time
from datetime import datetime
import boto3
import s3fs
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import joblib
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS


In [ ]:
# =====================
# CONFIG — EDIT ME
# =====================
CONFIG = {
    'AWS_REGION': os.environ.get('AWS_DEFAULT_REGION', 'us-east-1'),
    'S3_BUCKET': 'your-bucket-name',                 # <-- change
    'S3_INPUT_PREFIX': 'datasets/text/',             # S3 prefix containing one or more CSVs
    'S3_OUTPUT_PREFIX': f'model-artifacts/tfidf-xgb/{datetime.utcnow().strftime("%Y%m%d-%H%M%S")}/',
    'S3_GLOB': '*.csv',                              # pattern for multiple files under prefix
    'CHUNKED_READ': False,                           # True if each CSV is huge (set a chunksize below)
    'CHUNKSIZE': 500_000,                            # rows per chunk if CHUNKED_READ
    'RANDOM_STATE': 42,
    'TFIDF_MAX_FEATURES': 100_000,
    'TFIDF_MIN_DF': 2,
    'TFIDF_MAX_DF': 0.9,
    'TFIDF_NGRAMS': (1,2),
    'XGB_PARAMS': {
        'n_estimators': 600,
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'tree_method': 'hist',   # 'gpu_hist' if CUDA build available
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'random_state': 42,
    }
}
CONFIG

In [ ]:
# -----------------
# S3 utilities
# -----------------
s3 = boto3.client('s3', region_name=CONFIG['AWS_REGION'])
fs = s3fs.S3FileSystem(anon=False)

def s3_uri(bucket, key):
    return f's3://{bucket}/{key}'

def list_s3_csvs(bucket, prefix, pattern='*.csv'):
    base = s3_uri(bucket, prefix)
    return fs.glob(f'{base}/**/{pattern}') if pattern else fs.glob(f'{base}/**/*.csv')

def upload_to_s3(local_path, bucket, key):
    s3.upload_file(local_path, bucket, key)
    return s3_uri(bucket, key)


In [ ]:
# -----------------
# Load dataset(s)
# -----------------
csv_paths = list_s3_csvs(CONFIG['S3_BUCKET'], CONFIG['S3_INPUT_PREFIX'], CONFIG['S3_GLOB'])
assert csv_paths, f'No CSVs found at s3://{CONFIG["S3_BUCKET"]}/{CONFIG["S3_INPUT_PREFIX"]} matching {CONFIG["S3_GLOB"]}'
csv_paths[:5]

In [ ]:
# Concatenate all CSVs (schema must include: text,label)
dfs = []
for p in csv_paths:
    if CONFIG['CHUNKED_READ']:
        for chunk in pd.read_csv(p, storage_options={'anon': False}, chunksize=CONFIG['CHUNKSIZE']):
            dfs.append(chunk[['text','label']])
    else:
        df_i = pd.read_csv(p, storage_options={'anon': False})
        dfs.append(df_i[['text','label']])

df = pd.concat(dfs, axis=0, ignore_index=True)
df.shape

In [ ]:
# Hygiene
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 2].dropna(subset=['text','label'])
df = df.drop_duplicates(subset=['text','label'])
print('After hygiene:', df.shape)

# Label map
le = LabelEncoder()
y = le.fit_transform(df['label'])
X = df['text']
classes = np.unique(y)
print('Classes:', list(le.classes_))

In [ ]:
# Splits
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=CONFIG['RANDOM_STATE']
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=CONFIG['RANDOM_STATE']
)
len(X_train), len(X_val), len(X_test)

In [ ]:
# Minimal gensim clean for TF‑IDF
def clean_minimal(text: str) -> str:
    toks = [t for t in simple_preprocess(str(text), deacc=True) if t not in STOPWORDS]
    return ' '.join(toks)

X_train_c = X_train.apply(clean_minimal)
X_val_c   = X_val.apply(clean_minimal)
X_test_c  = X_test.apply(clean_minimal)

tfidf = TfidfVectorizer(
    min_df=CONFIG['TFIDF_MIN_DF'],
    max_df=CONFIG['TFIDF_MAX_DF'],
    ngram_range=tuple(CONFIG['TFIDF_NGRAMS']),
    max_features=CONFIG['TFIDF_MAX_FEATURES'],
)
Xtr = tfidf.fit_transform(X_train_c)
Xva = tfidf.transform(X_val_c)
Xte = tfidf.transform(X_test_c)
Xtr.shape, Xva.shape, Xte.shape

In [ ]:
# Class weights → per-sample weights
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, cw)}
sample_w = np.array([class_weight_dict[int(t)] for t in y_train])
class_weight_dict

In [ ]:
# Train XGBoost
params = dict(CONFIG['XGB_PARAMS'])
params['num_class'] = len(classes)
clf = XGBClassifier(**params)
clf.fit(Xtr, y_train, sample_weight=sample_w, eval_set=[(Xva, y_val)], verbose=False)

y_pred = clf.predict(Xte)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print('macro-F1:', f1_score(y_test, y_pred, average='macro'))

In [ ]:
# Save locally
ART_DIR = f'tfidf_xgb_artifacts_{datetime.utcnow().strftime("%Y%m%d-%H%M%S")}'
os.makedirs(ART_DIR, exist_ok=True)
joblib.dump(clf, f'{ART_DIR}/xgb_model.joblib')
joblib.dump(tfidf, f'{ART_DIR}/tfidf_vectorizer.joblib')
joblib.dump(le, f'{ART_DIR}/label_encoder.joblib')
with open(f'{ART_DIR}/classes.json','w') as f:
    json.dump(list(le.classes_), f)
print('Saved locally to', ART_DIR)

In [ ]:
# Upload artifacts to S3
uploaded = []
for fname in ['xgb_model.joblib','tfidf_vectorizer.joblib','label_encoder.joblib','classes.json']:
    key = CONFIG['S3_OUTPUT_PREFIX'] + fname
    uri = upload_to_s3(f'{ART_DIR}/{fname}', CONFIG['S3_BUCKET'], key)
    uploaded.append(uri)
uploaded